In [ ]:
import pandas as pd
import calendar
import re

# ============================================
# FILE PATHS
# ============================================
indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file = r"D:/Tushar/main_with_subs_only.xlsx"

# ============================================
# READ FILES
# ============================================
indent_df = pd.read_excel(indent_file)
bom_df = pd.read_excel(bom_file)

# ============================================
# FIND LATEST MONTH COLUMN
# ============================================
month_pattern = re.compile(r"[A-Za-z]{3}'?\d{2}")

month_cols = [col for col in indent_df.columns if month_pattern.search(str(col))]

latest_col = month_cols[-1]

print("Using month column:", latest_col)

# ============================================
# EXTRACT MONTH + YEAR
# ============================================
match = re.search(r"([A-Za-z]{3})'(\d{2})", latest_col)

month_str = match.group(1)
year = int("20" + match.group(2))

month_num = list(calendar.month_abbr).index(month_str)

days_in_month = calendar.monthrange(year, month_num)[1]

# ============================================
# PREPARE INDENT DATA
# ============================================
indent_df = indent_df[['Part number', latest_col]].dropna()

indent_df.rename(columns={
    'Part number': 'Switch',
    latest_col: 'Monthly_Qty'
}, inplace=True)

indent_df['Daily_Qty'] = indent_df['Monthly_Qty'] / days_in_month

# ============================================
# PREPARE BOM DATA
# ============================================
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']]

bom_df.rename(columns={
    'Main_Label': 'Child',
    'Sub_Label': 'Switch',
    'Sub_Count': 'Usage_Qty'
}, inplace=True)

# ============================================
# MERGE
# ============================================
merged = pd.merge(bom_df, indent_df, on='Switch', how='inner')

merged['Daily_Child'] = merged['Daily_Qty'] * merged['Usage_Qty']
merged['Two_Day_Qty'] = merged['Daily_Child'] * 2

# ============================================
# AGGREGATE
# ============================================
result = merged.groupby('Child')['Two_Day_Qty'].sum().reset_index()

print(result)

# Optional save
result.to_excel("Two_Day_Child_Requirement.xlsx", index=False)
